In [1]:
from mlds.data_loader import CLINC150_MAPPER, INTENTS
from datasets import load_dataset
import pandas as pd

def load():
    dfs = {}
    for split in ["train", "validation", "test"]:
        # 100, 20, 30
        dataset = load_dataset("clinc/clinc_oos", "plus", split=split)
        df = dataset.to_pandas()
        df["intent"] = df["intent"].map(CLINC150_MAPPER)
        df = df[df["intent"].isin(INTENTS)]
        dfs[split] = df
    # print(df["intent"].value_counts())
    return dfs

dfs = load()

In [2]:
df = dfs["train"]
records = []
for intent in INTENTS:
    intent_df = df[df["intent"] == intent].head(80)
    print(intent, len(intent_df))
    records.append(intent_df)

df = pd.concat(records)
df = df[["intent", "text"]]
df["domain"] = "clinc"
df["spans"] = ""
df["logical_form"] = "[SL: " + df["intent"] + "]"
# split,domain,intent,text,spans,logical_form
# 1047
df.to_csv("data/output/clinc.csv", index=False)
df.to_csv("data/output/clinc+extend.csv", index=False)

alarm 80
balance 80
bill_balance 80
book_flight 80
book_hotel 80
calendar_update 80
cancel_reservation 80
car_rental 80
confirm_reservation 80
cook_time 80
exchange_rate 80
food_last 80
freeze_account 80
ingredients_list 80
interest_rate 80
international_visa 80
make_call 80
meal_suggestion 80
min_payment 80
pay_bill 80
pin_change 80
play_music 80
plug_type 80
recipe 80
restaurant_reservation 80
restaurant_reviews 80
restaurant_suggestion 80
share_location 80
shopping_list_update 80
spending_history 80
text 80
time 80
timezone 80
transactions 80
transfer 80
translate 80
travel_notification 80
travel_suggestion 80
update_playlist 80
weather 80


In [3]:
for shot in [(5, 1, 2), (10, 2, 3), (25, 5, 8), (50, 10, 15), (100, 20, 30)]:
    file = []
    for split in ["train", "validation", "test"]:
        size = shot[["train", "validation", "test"].index(split)]
        df = dfs[split]
        records = []
        for intent in INTENTS:
            intent_df = df[df["intent"] == intent].head(size)
            records.append(intent_df)

        df = pd.concat(records)
        df = df[["intent", "text"]]
        df["domain"] = "clinc"
        df["spans"] = ""
        df["logical_form"] = "[SL: " + df["intent"] + "]"
        df["split"] = split if split != "validation" else "dev"
        file.append(df)

    df = pd.concat(file)
    df.to_csv("data/output/clinc_{}shots.csv".format(shot[0]), index=False)
    print(df["split"].value_counts())
    print(df["intent"].value_counts())

split
train    200
test      80
dev       40
Name: count, dtype: int64
intent
alarm                     8
balance                   8
plug_type                 8
recipe                    8
restaurant_reservation    8
restaurant_reviews        8
restaurant_suggestion     8
share_location            8
shopping_list_update      8
spending_history          8
text                      8
time                      8
timezone                  8
transactions              8
transfer                  8
translate                 8
travel_notification       8
travel_suggestion         8
update_playlist           8
play_music                8
pin_change                8
pay_bill                  8
cook_time                 8
bill_balance              8
book_flight               8
book_hotel                8
calendar_update           8
cancel_reservation        8
car_rental                8
confirm_reservation       8
exchange_rate             8
min_payment               8
food_last                 

In [4]:
from mlds.data_loader import SlotDataManager, INTENTS
slot_manager = SlotDataManager(data_folder="data/output")
eng = slot_manager.load_data("eng", "split")

Missing balance with {'lug', 'sna'}
Missing confirm_reservation with {'lug', 'sna'}
Missing freeze_account with {'lug', 'sna'}
Missing restaurant_reservation with {'lug', 'sna'}
Missing shopping_list_update with {'lug', 'sna'}
Missing time with {'lug', 'sna'}
Missing timezone with {'lug', 'sna'}
Missing transfer with {'lug', 'sna'}
Missing translate with {'lug', 'sna'}


In [5]:
for split in ["train", "dev", "test"]:
    print(eng[split]["intent"].value_counts())

intent
recipe                    28
ingredients_list          28
alarm                     28
cook_time                 28
food_last                 28
pay_bill                  28
restaurant_suggestion     28
weather                   28
international_visa        28
bill_balance              28
car_rental                28
cancel_reservation        28
update_playlist           28
calendar_update           28
play_music                28
share_location            28
text                      28
book_hotel                28
plug_type                 28
make_call                 28
travel_notification       28
travel_suggestion         28
spending_history          28
pin_change                28
interest_rate             27
min_payment               27
exchange_rate             27
book_flight               27
restaurant_reviews        27
meal_suggestion           27
transactions              25
translate                 23
timezone                  21
freeze_account            21
restaur

In [ ]:

# eng["dev"] = pd.concat([eng["dev"], eng["test"]])


def prepare_df(records, split):
    df = pd.concat(records)
    df = df[["intent", "text"]]
    df["domain"] = "eng"
    df["spans"] = ""
    df["logical_form"] = "[SL: " + df["intent"] + "]"
    df["split"] = split
    return df
# 
for shot in [(5, 1, 2), (10, 2, 3), (25, 5, 8)]:
    file = []

    # dev
    split = "dev"
    size = shot[1]
    df = pd.concat([eng["dev"], eng["test"]])
    records = []
    for intent in INTENTS:
        sliced_df = df[df["intent"] == intent].head(size)
        records.append(sliced_df)

        if len(sliced_df) < size:
            print(split, intent, len(sliced_df))

    file.append(prepare_df(records, split))

    # test
    split = "test"
    size = shot[2]
    df = eng[split]
    records = []
    train_additional = []
    for intent in INTENTS:
        sliced_df = df[df["intent"] == intent].tail(size)
        records.append(sliced_df)
        train_additional.append(df[df["intent"] == intent].tail(size + 5).head(5))
        if len(sliced_df) < size:
            print(split, intent, len(sliced_df))

    file.append(prepare_df(records, split))

    # train
    split = "train"
    size = shot[0]
    # df = eng[split]
    df = pd.concat([eng[split]] + train_additional)
    records = []
    for intent in INTENTS:
        sliced_df = df[df["intent"] == intent].head(size)
        records.append(sliced_df)

        if len(sliced_df) < size:
            print(split, intent, len(sliced_df))
            
    print(records[-1].shape)
    file.append(prepare_df(records, split))

    df = pd.concat(file)
    df.to_csv("data/output/eng_{}shots.csv".format(shot[0]), index=False)
    print(df["split"].value_counts())
    print(df["intent"].value_counts())


(5, 9)
split
train    200
test      80
dev       40
Name: count, dtype: int64
intent
alarm                     8
balance                   8
plug_type                 8
recipe                    8
restaurant_reservation    8
restaurant_reviews        8
restaurant_suggestion     8
share_location            8
shopping_list_update      8
spending_history          8
text                      8
time                      8
timezone                  8
transactions              8
transfer                  8
translate                 8
travel_notification       8
travel_suggestion         8
update_playlist           8
play_music                8
pin_change                8
pay_bill                  8
cook_time                 8
bill_balance              8
book_flight               8
book_hotel                8
calendar_update           8
cancel_reservation        8
car_rental                8
confirm_reservation       8
exchange_rate             8
min_payment               8
food_last          